In [11]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from PIL import Image
import os

def create_heatmap_gif(array1, array2, 
                      title1='Heatmap 1', title2='Heatmap 2', 
                      cmap='viridis', figsize=(18, 6),
                      colorbar=True, 
                      fps=10, dpi=100, text="training"):
    # Create temporary directory for frames
    os.makedirs('temp_frames', exist_ok=True)
    
    frame_files = []
    
    for idx in range(array1.shape[0]):
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=figsize)
        
        # Plot first heatmap
        im1 = ax1.imshow(array1[idx], cmap=cmap, origin='lower')
        ax1.set_title(title1)
        ax1.set_xlabel('X')
        ax1.set_ylabel('Y')
        if colorbar:
            fig.colorbar(im1, ax=ax1, label='Value')
        
        # Plot second heatmap
        im2 = ax2.imshow(array2[idx], cmap=cmap, origin='lower')
        ax2.set_title(title2)
        ax2.set_xlabel('X')
        ax2.set_ylabel('Y')
        if colorbar:
            fig.colorbar(im2, ax=ax2, label='Value')

        # Plot second heatmap
        im3 = ax3.imshow(array2[idx]-array1[idx], cmap="coolwarm", origin='lower')
        ax3.set_title(f"{title2}-{title1} difference")
        ax3.set_xlabel('X')
        ax3.set_ylabel('Y')
        if colorbar:
            fig.colorbar(im3, ax=ax3, label='Value')
        
        plt.suptitle(f"Navier-Stokes Vorticity Transport (frame {(idx+1)*2+20}) ({text})")
        plt.tight_layout()
        
        # Save frame
        frame_file = f"temp_frames/frame_{idx:04d}.png"
        plt.savefig(frame_file, dpi=dpi, bbox_inches='tight')
        frame_files.append(frame_file)
        plt.close()
    
    # Create GIF from frames
    images = [Image.open(f) for f in frame_files]
    images[0].save(f"{text}.gif", 
                  save_all=True, 
                  append_images=images[1:], 
                  duration=1000//fps, 
                  loop=0)
    
    # Clean up temporary files
    for f in frame_files:
        os.remove(f)
    os.rmdir('temp_frames')
    
    print(f"GIF saved as {text}.gif")

In [2]:
import os
import sys
from pathlib import Path
import torch

notebook_path = Path(os.getcwd())
project_root = notebook_path.parent.parent.parent
sys.path.append(str(project_root))

from models.FNO2d import FNO2d

# Load the data file
data_test = torch.load('../../../datasets/2D/NS/dim2d_nx256_N200_solver=exponax_kernel=matern_correlation_length0.50_bcperiodic_nu0.010_t40.0_seed2000_all_frames.pt', weights_only=False)
model_instance = FNO2d(
    modes1=128, 
    modes2=128, 
    width=64
)
state_dict = torch.load(
    'NS_2d_FNO_model_trainedby_dim2d_nx256_N1500_solver=exponax_kernel=matern_correlation_length0.50_bcperiodic_nu0.010_t40.0_seed0_all_frames.pth',
    map_location=torch.device('cpu')
)
model_instance.load_state_dict(state_dict)
model_instance.eval() 

FNO2d(
  (p): Linear(in_features=12, out_features=64, bias=True)
  (conv_layers): ModuleList(
    (0-3): 4 x SpectralConv2d()
  )
  (mlp_layers): ModuleList(
    (0-3): 4 x MLP(
      (mlp): Sequential(
        (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
        (1): GELU(approximate='none')
        (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
      )
    )
  )
  (w_layers): ModuleList(
    (0-3): 4 x Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
  )
  (q): MLP(
    (mlp): Sequential(
      (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
      (1): GELU(approximate='none')
      (2): Conv2d(256, 1, kernel_size=(1, 1), stride=(1, 1))
    )
  )
)

In [12]:
import numpy as np


def plot_gifs(idx):
    prediction_steps = 20
    T_in = 10
    T = 10

    with torch.no_grad():
        # Prepare initial input for model (add batch and channel dims)
        current_input = data_test["u"][idx].permute(1,2,0).unsqueeze(0)[...,::2][...,:T_in]  # Shape: (1, x, y, T_in)
        
        # Get ground truth (for comparison)
        ground_truth = data_test["u"][idx].permute(1,2,0).unsqueeze(0)[...,::2][...,T_in:T+T_in]  # Shape: (1, x, y, T)
        
        # Collect predictions
        predictions = []
        
        # Autoregressive prediction loop
        for step in range(prediction_steps-10):
            # Forward pass
            pred = model_instance(current_input)
            
            # Remove batch and channel dims, keep only next step prediction
            pred_squeezed = pred.squeeze()  # Shape: (x, y)
            predictions.append(pred_squeezed.numpy())
            
            # Update input: remove oldest step, add new prediction
            current_input = torch.cat([
                current_input[..., 1:],  # Remove oldest step
                pred         # Add new prediction
            ], dim=-1)
        
        # Convert to numpy arrays
        predictions = np.stack(predictions)  # Shape: (prediction_steps, x, y)
        ground_truth = ground_truth.squeeze(0).numpy().transpose(2, 0, 1)  # Shape: (T, x, y)
        
        # Visualize first 10 steps of ground truth vs predictions
        create_heatmap_gif(
            ground_truth[:10],  # First 10 steps of ground truth
            predictions[:10],   # First 10 steps of predictions
            title1='PDE Solution',
            title2='FNO Prediction',
            fps=5,
            text=f"modes128-width64 test (Autoregressive 20-40 steps prediction) index={idx}"
        )

for idx in range(3):
    plot_gifs(idx)

GIF saved as modes128-width64 test (Autoregressive 20-40 steps prediction) index=0.gif
GIF saved as modes128-width64 test (Autoregressive 20-40 steps prediction) index=1.gif
GIF saved as modes128-width64 test (Autoregressive 20-40 steps prediction) index=2.gif


In [5]:
current_input.shape

torch.Size([1, 41, 256, 10])

In [20]:
predictions[:10].shape

(10, 1, 64, 64)

In [19]:
ground_truth[:10].shape

(10, 64, 64)

In [14]:
pred.shape

torch.Size([1, 64, 64, 1])

In [8]:
data_test["x"].shape

torch.Size([50, 64, 64])

In [7]:
data_test["x"][idx].shape

torch.Size([64, 64])